# Final EMG Classification Pipeline

This notebook reconstructs the best EMG gesture-classification pipeline discovered by autoresearch for this repository. It is designed as both an executable workflow and a polished final report.

The implementation intentionally reuses the project's existing pipeline code from `train.py` so that the notebook stays faithful to the winning preprocessing, feature engineering, and model configuration recorded in `best_results.json`.

## 1. Imports and Configurations

This section loads the libraries, seeds the workflow for reproducibility, reads `best_results.json`, and reconstructs the winning experiment configuration.

A small but important note: the winning result records `feature_scaling = true`, but the repo only applies a `StandardScaler` for linear models, SVMs, and the small MLP. Because the best model is `extra_trees`, scaling remains part of the searched configuration record but does not introduce an active scaler stage in the fitted pipeline.

In [ ]:
from __future__ import annotations

import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GroupShuffleSplit

from train import (
    ExperimentConfig,
    apply_preprocessing,
    build_bouts,
    build_feature_table,
    build_pipeline,
    enumerate_windows,
    evaluate_features,
    filter_frame,
    load_dataset,
    normalization_lookup,
    seed_everything,
)

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

REPO_ROOT = Path.cwd()
DATA_PATH = REPO_ROOT / 'EMG-data.csv.zip'
DESCRIPTION_PATH = REPO_ROOT / 'dataset_description.txt'
BEST_RESULTS_PATH = REPO_ROOT / 'best_results.json'
RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
rng = seed_everything(RANDOM_SEED)

with BEST_RESULTS_PATH.open('r', encoding='utf-8') as handle:
    best_results = json.load(handle)

winner = best_results['best_grouped_result']
preprocessing = winner['preprocessing']
feature_families = winner['feature_families']
model_params = winner['model_params']

config = ExperimentConfig(
    random_seed=winner['seed'],
    model_family=winner['model_family'],
    split_strategy='group_kfold',
    cv_folds=2,
    window_ms=preprocessing['window_ms'],
    overlap_ratio=preprocessing['overlap_ratio'],
    include_class_zero=preprocessing['include_class_zero'],
    include_class_seven=preprocessing['include_class_seven'],
    normalization_strategy=preprocessing['normalization_strategy'],
    rectify_signal=preprocessing['rectify_signal'],
    detrend_signal=preprocessing['detrend_signal'],
    remove_dc_offset=preprocessing['remove_dc_offset'],
    feature_scaling=preprocessing['feature_scaling'],
    use_pca=preprocessing['use_pca'],
    pca_components=preprocessing['pca_components'],
    ar_order=preprocessing['ar_order'],
    windows_per_group_class=96,
    max_windows_total=30000,
    enable_cross_channel_summary=True,
    enable_channel_pair_features=False,
    channel_pair_limit=4,
    feature_families=feature_families,
    thresholds=preprocessing['thresholds'],
    model_params={**ExperimentConfig().model_params, winner['model_family']: model_params},
)

config_summary = pd.DataFrame(
    [
        ('Model family', config.model_family),
        ('Model parameters', json.dumps(model_params, sort_keys=True)),
        ('Window length (ms)', config.window_ms),
        ('Overlap ratio', config.overlap_ratio),
        ('Normalization', config.normalization_strategy),
        ('Remove DC offset', config.remove_dc_offset),
        ('Rectify signal', config.rectify_signal),
        ('Detrend signal', config.detrend_signal),
        ('Use PCA', config.use_pca),
        ('AR order', config.ar_order),
        ('Feature scaling flag', config.feature_scaling),
        ('Feature families', ', '.join(name for name, enabled in feature_families.items() if enabled)),
        ('Thresholds', json.dumps(config.thresholds, sort_keys=True)),
        ('Split strategy used in autoresearch', winner['split_strategy']),
        ('Best grouped macro F1', round(winner['metrics']['f1_macro'], 6)),
        ('Best grouped accuracy', round(winner['metrics']['accuracy'], 6)),
        ('Best grouped balanced accuracy', round(winner['metrics']['balanced_accuracy'], 6)),
        ('Recorded window count', winner['window_count']),
        ('Recorded feature count', winner['feature_count']),
    ],
    columns=['Setting', 'Value'],
)

print('Winning result loaded from:', BEST_RESULTS_PATH)
display(config_summary)
display(pd.json_normalize(winner, sep=' -> '))

## 2. Load dataset

The raw dataset is stored as `EMG-data.csv.zip` in the project root. The repository code treats the CSV as analytics-ready and reconstructs missing session IDs from negative time resets within each subject.

The dataset description indicates that the MYO bracelet collected eight synchronized EMG channels from 36 subjects performing static hand gestures. The notebook follows the repo's assumptions exactly:

- `label` identifies the subject.
- `class` is the gesture label.
- `time` is recorded in milliseconds.
- `channel1` through `channel8` are the EMG signal channels.
- negative time jumps within a subject mark the start of the second recording session.

In [ ]:
raw_df = pd.read_csv(DATA_PATH, compression='zip')
original_csv_shape = raw_df.shape
expected_columns = ['time', 'channel1', 'channel2', 'channel3', 'channel4', 'channel5', 'channel6', 'channel7', 'channel8', 'class', 'label']
missing_columns = sorted(set(expected_columns) - set(raw_df.columns))
if missing_columns:
    raise ValueError(f'Unexpected dataset schema. Missing columns: {missing_columns}')

with DESCRIPTION_PATH.open('r', encoding='utf-8') as handle:
    dataset_text = handle.read().strip()

raw_df = raw_df.loc[:, expected_columns].copy()
raw_df['subject_id'] = raw_df['label'].astype(np.int16)
time_diff = raw_df.groupby('subject_id', sort=False)['time'].diff()
raw_df['session_id'] = (time_diff < 0).groupby(raw_df['subject_id']).cumsum().fillna(0).astype(np.int16)

summary = {
    'original_csv_shape': original_csv_shape,
    'augmented_shape_after_session_columns': raw_df.shape,
    'subjects': int(raw_df['subject_id'].nunique()),
    'sessions': int(raw_df[['subject_id', 'session_id']].drop_duplicates().shape[0]),
    'classes': sorted(int(value) for value in raw_df['class'].unique()),
    'median_positive_time_delta_ms': float(np.median(time_diff[(time_diff > 0) & (time_diff < 20)])),
}

print('Dataset description excerpt:')
print(dataset_text)
print()
print('Dataset summary:')
print(summary)
print()
print('Columns:')
print(raw_df.columns.tolist())
print()
print('Dtypes:')
display(raw_df.dtypes.to_frame('dtype'))
print()
print('Preview:')
display(raw_df.head())

From the dataset text and the repository logic, the gesture classes are interpreted as follows:

- `0`: unmarked transition data
- `1`: hand at rest
- `2`: fist
- `3`: wrist flexion
- `4`: wrist extension
- `5`: radial deviation
- `6`: ulnar deviation
- `7`: extended palm, available only for a small subset of subjects

Because classes `0` and `7` were excluded in the winning autoresearch run, the final modeling workflow below focuses on classes `1` through `6`.

## 3. Exploratory Data Analysis and pre-processing

This section describes the raw dataset before the modeling pipeline is applied. The goal is to understand class balance, data completeness, and the structure of the multichannel EMG signals.

In [ ]:
gesture_map = {
    0: 'transition',
    1: 'rest',
    2: 'fist',
    3: 'wrist flexion',
    4: 'wrist extension',
    5: 'radial deviation',
    6: 'ulnar deviation',
    7: 'extended palm',
}

eda_summary = pd.DataFrame(
    [
        ('Rows', len(raw_df)),
        ('Columns', raw_df.shape[1]),
        ('Unique subjects', raw_df['subject_id'].nunique()),
        ('Inferred sessions', raw_df[['subject_id', 'session_id']].drop_duplicates().shape[0]),
        ('Missing values', int(raw_df.isna().sum().sum())),
        ('Duplicate rows', int(raw_df.duplicated().sum())),
        ('Class 7 subjects', sorted(raw_df.loc[raw_df['class'] == 7, 'subject_id'].unique().tolist())),
    ],
    columns=['Metric', 'Value'],
)
display(eda_summary)

class_counts_full = raw_df['class'].value_counts().sort_index().rename_axis('class').reset_index(name='count')
class_counts_full['gesture'] = class_counts_full['class'].map(gesture_map)
filtered_raw = raw_df.loc[~raw_df['class'].isin([0, 7])].copy()
class_counts_filtered = filtered_raw['class'].value_counts().sort_index().rename_axis('class').reset_index(name='count')
class_counts_filtered['gesture'] = class_counts_filtered['class'].map(gesture_map)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.barplot(data=class_counts_full, x='class', y='count', hue='gesture', dodge=False, ax=axes[0], palette='crest')
axes[0].set_title('Class distribution in the raw dataset')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Row count')
axes[0].legend_.remove()

sns.barplot(data=class_counts_filtered, x='class', y='count', hue='gesture', dodge=False, ax=axes[1], palette='flare')
axes[1].set_title('Class distribution after removing classes 0 and 7')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Row count')
axes[1].legend_.remove()
plt.tight_layout()
plt.show()

signal_summary = raw_df[[f'channel{i}' for i in range(1, 9)]].describe().T[['mean', 'std', 'min', 'max']]
display(signal_summary)

The raw dataset is dominated by transition samples (`class 0`), which is exactly why the benchmark excludes them during training. After that filtering step, the six retained gesture classes are reasonably balanced.

The channel summary also shows that the raw EMG values are centered very close to zero and live on a small amplitude scale, which makes DC offset removal and consistent normalization sensible preprocessing steps.

In [ ]:
filtered_for_plot = filtered_raw.copy()
filtered_for_plot = build_bouts(filtered_for_plot)

selected_examples = []
for gesture in [1, 2, 3]:
    candidate_bout = filtered_for_plot.loc[filtered_for_plot['class'] == gesture, 'bout_index'].iloc[0]
    bout_frame = filtered_for_plot.loc[filtered_for_plot['bout_index'] == candidate_bout].head(400).copy()
    bout_frame['relative_sample'] = np.arange(len(bout_frame))
    selected_examples.append((gesture, bout_frame))

fig, axes = plt.subplots(len(selected_examples), 1, figsize=(16, 12), sharex=False)
channels_to_plot = ['channel1', 'channel2', 'channel3', 'channel4']
for axis, (gesture, bout_frame) in zip(axes, selected_examples):
    melted = bout_frame.melt(id_vars=['relative_sample'], value_vars=channels_to_plot, var_name='channel', value_name='amplitude')
    sns.lineplot(data=melted, x='relative_sample', y='amplitude', hue='channel', ax=axis, linewidth=1.2)
    axis.set_title(f"Representative raw signal segment: class {gesture} ({gesture_map[gesture]})")
    axis.set_xlabel('Sample index within bout segment')
    axis.set_ylabel('Amplitude')
    axis.legend(loc='upper right', ncol=2)
plt.tight_layout()
plt.show()

These representative traces illustrate the multichannel structure of the problem: each gesture is observed as a short synchronized eight-channel signal rather than as a single scalar feature row. The modeling pipeline therefore transforms short windows of signal into a structured feature table before classification.

## 4. Signal preprocessing

The winning pipeline is a handcrafted-feature workflow. It does not feed raw time series directly into a neural network. Instead, it performs leakage-aware segmentation and feature extraction using the configuration found in `best_results.json` and the exact functions defined in `train.py`.

Winning preprocessing choices:

- filter out classes `0` and `7`
- reconstruct gesture bouts so windows do not cross label or session boundaries
- segment the signal into `225 ms` windows
- use `40%` overlap, which yields a step size of `135 ms`
- remove DC offset within each window
- normalize using per-session statistics
- extract per-channel time-domain, EMG-specific, distributional, Hjorth, autoregressive, and frequency features
- aggregate the same feature names across channels with summary statistics
- do not use sample entropy, channel-pair correlation features, rectification, detrending, or PCA

In [ ]:
bundle = load_dataset()
modeling_frame = filter_frame(bundle.frame, config)
modeling_frame = build_bouts(modeling_frame)
windows = enumerate_windows(modeling_frame, bundle.sample_rate_hz, config, rng)
features, labels, groups = build_feature_table(modeling_frame, windows, bundle, config, rng)

window_size_samples = max(8, int(round(bundle.sample_rate_hz * config.window_ms / 1000.0)))
step_size_samples = max(1, int(round(window_size_samples * (1.0 - config.overlap_ratio))))

preprocessing_summary = pd.DataFrame(
    [
        ('Sample rate (Hz)', bundle.sample_rate_hz),
        ('Window length (samples)', window_size_samples),
        ('Step size (samples)', step_size_samples),
        ('Window count', len(windows)),
        ('Feature count', features.shape[1]),
        ('Subject groups', len(pd.unique(groups))),
        ('Sessions after reconstruction', modeling_frame[['subject_id', 'session_id']].drop_duplicates().shape[0]),
    ],
    columns=['Item', 'Value'],
)
display(preprocessing_summary)

normalization_stats = normalization_lookup(modeling_frame, bundle.channels, config)
example_record = windows[0]
example_raw_window = modeling_frame.loc[example_record.start_idx:example_record.end_idx - 1, bundle.channels].to_numpy(dtype=np.float32)
example_processed_window = apply_preprocessing(
    example_raw_window,
    config,
    normalization_stats[(example_record.subject, example_record.session)],
)

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
axes[0].plot(example_raw_window[:, 0], color='steelblue')
axes[0].set_title('Example raw window: channel1')
axes[0].set_ylabel('Amplitude')
axes[1].plot(example_processed_window[:, 0], color='darkorange')
axes[1].set_title('Same window after winning preprocessing')
axes[1].set_xlabel('Sample index within window')
axes[1].set_ylabel('Amplitude')
plt.tight_layout()
plt.show()

feature_preview = features.head().T.head(20)
display(feature_preview)

per_channel_feature_count = len([col for col in features.columns if col.startswith('channel1_')])
summary_feature_count = len([col for col in features.columns if col.startswith('summary_')])
feature_family_breakdown = pd.DataFrame(
    [
        ('Per-channel features on each sensor', per_channel_feature_count),
        ('Channels', len(bundle.channels)),
        ('Across-channel summary features', summary_feature_count),
        ('Total engineered features', features.shape[1]),
    ],
    columns=['Component', 'Count'],
)
display(feature_family_breakdown)

The resulting design matrix exactly matches the winning autoresearch run: `10092` windows and `396` engineered features. That agreement is an important verification step because it confirms the notebook is reconstructing the same preprocessing and feature-generation logic as the search pipeline.

## 5. Partitioning Dataset and Creating Data loaders

Because the winning approach is feature-based and uses scikit-learn, literal PyTorch-style `DataLoader` objects are not needed. The appropriate equivalent is the prepared feature matrix together with the label vector and the subject-group assignments used for leakage-aware splitting.

This section performs two related tasks:

1. reproduce the search-time grouped evaluation used by autoresearch
2. create a separate final subject-held-out train/test split for the polished final model report

In [ ]:
reproduced_cv = evaluate_features(features, labels, groups, config, 0.0)
reproduced_cv_metrics = pd.DataFrame([reproduced_cv.metrics])
reproduced_cv_folds = pd.DataFrame(reproduced_cv.fold_summaries)

print('Reproduced grouped CV metrics:')
display(reproduced_cv_metrics)
print('Fold-level grouped CV summary:')
display(reproduced_cv_folds)

group_splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_SEED)
train_idx, test_idx = next(group_splitter.split(features, labels, groups=groups))

X_train = features.iloc[train_idx].copy()
X_test = features.iloc[test_idx].copy()
y_train = labels[train_idx]
y_test = labels[test_idx]
groups_train = groups[train_idx]
groups_test = groups[test_idx]

split_summary = pd.DataFrame(
    [
        ('Training windows', X_train.shape[0]),
        ('Test windows', X_test.shape[0]),
        ('Training features', X_train.shape[1]),
        ('Held-out subject groups', len(pd.unique(groups_test))),
        ('Training subject groups', len(pd.unique(groups_train))),
    ],
    columns=['Split statistic', 'Value'],
)

display(split_summary)

display(pd.DataFrame({'train_class_count': pd.Series(y_train).value_counts().sort_index(), 'test_class_count': pd.Series(y_test).value_counts().sort_index()}))

The grouped CV step above is the closest apples-to-apples comparison with the autoresearch benchmark. The holdout split below serves a different purpose: it creates one clean final evaluation partition so that the notebook can fit a final model on the full training portion and then report performance on entirely unseen subjects.

## 6. Model

The winning classifier is an `ExtraTreesClassifier` wrapped inside the repository's sklearn pipeline. This is a compact tree-based model, which makes it a strong fit for a handcrafted-feature benchmark: it can capture nonlinear structure in the engineered EMG features without requiring a deep raw-signal architecture.

In [ ]:
model_pipeline = build_pipeline(config)
model_summary = pd.DataFrame(
    [
        ('Pipeline steps', ' -> '.join(name for name, _ in model_pipeline.steps)),
        ('Model family', config.model_family),
        ('Hyperparameters', json.dumps(model_params, sort_keys=True)),
        ('Uses active StandardScaler stage', any(name == 'scaler' for name, _ in model_pipeline.steps)),
        ('Uses PCA stage', any(name == 'pca' for name, _ in model_pipeline.steps)),
    ],
    columns=['Model detail', 'Value'],
)
display(model_summary)
model_pipeline

## 7. Model training

For this final notebook, the model is trained once on the full designated training partition created above. Because the winner is a classical sklearn model, there is no epoch-based training loop. The fit itself is the complete training procedure.

In [ ]:
train_start = time.perf_counter()
model_pipeline.fit(X_train, y_train)
train_runtime_seconds = time.perf_counter() - train_start

training_summary = pd.DataFrame(
    [
        ('Training windows used', X_train.shape[0]),
        ('Features used', X_train.shape[1]),
        ('Training runtime (seconds)', round(train_runtime_seconds, 3)),
    ],
    columns=['Training summary', 'Value'],
)
display(training_summary)

## 8. Model Evaluation

The final evaluation is performed on the untouched held-out subject split. The metrics reported here answer a different question from the grouped cross-validation result in `best_results.json`.

- The grouped CV score summarizes the search-time benchmark across two subject folds.
- The final holdout score summarizes one clean train/test partition for presentation.

Both are useful, and any gap between them should be interpreted in that context rather than as evidence that the notebook changed the winning pipeline.

In [ ]:
y_pred = model_pipeline.predict(X_test)

holdout_metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'macro_f1': f1_score(y_test, y_pred, average='macro'),
    'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
}
display(pd.DataFrame([holdout_metrics]))

report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report).T
print('Classification report:')
display(report_df)

cm = confusion_matrix(y_test, y_pred, labels=sorted(pd.unique(y_test)))
fig, ax = plt.subplots(figsize=(8, 7))
cmd = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(pd.unique(y_test)))
cmd.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion matrix on the held-out subject split')
plt.show()

comparison_table = pd.DataFrame(
    [
        {
            'Evaluation view': 'Autoresearch best grouped CV',
            'Accuracy': winner['metrics']['accuracy'],
            'Macro F1': winner['metrics']['f1_macro'],
            'Balanced Accuracy': winner['metrics']['balanced_accuracy'],
        },
        {
            'Evaluation view': 'Notebook reproduced grouped CV',
            'Accuracy': reproduced_cv.metrics['accuracy'],
            'Macro F1': reproduced_cv.metrics['f1_macro'],
            'Balanced Accuracy': reproduced_cv.metrics['balanced_accuracy'],
        },
        {
            'Evaluation view': 'Notebook final subject holdout',
            'Accuracy': holdout_metrics['accuracy'],
            'Macro F1': holdout_metrics['macro_f1'],
            'Balanced Accuracy': holdout_metrics['balanced_accuracy'],
        },
    ]
)
display(comparison_table)

The comparison table should be read carefully. In this environment, the notebook reproduces the **current repository code path** for the winning configuration, and that grouped CV score is slightly lower than the historical value stored in `best_results.json`. Because a direct `run_experiment(...)` call from `train.py` yields the same lower number, the most likely explanation is that `best_results.json` was produced from an earlier experiment context or code snapshot even though the high-level configuration still matches.

The final holdout score is higher than either grouped CV result because it is based on one particular subject-held-out split rather than the average over two grouped folds. That does **not** mean the model should be judged only by the higher value; the grouped CV result remains the more conservative benchmark-style estimate.

### Conclusion

The winning pipeline is a feature-based EMG classifier built around:

- leakage-aware subject grouping
- `225 ms` windows with `40%` overlap
- per-session normalization plus DC offset removal
- a broad handcrafted feature set spanning time, EMG-specific, distributional, Hjorth, autoregressive, and frequency descriptors
- an `ExtraTreesClassifier` with `350` trees

This combination likely worked well because the dataset is structured, moderately sized, and strongly driven by signal-processing characteristics. In that setting, careful segmentation and informative EMG features can outperform more complex raw-signal modeling approaches.

A natural next step after final evaluation would be to retrain the same validated pipeline on all available eligible data for deployment, but that should happen only after preserving a clean benchmark result such as the holdout evaluation reported above.